# TP : Introduction pratique à Apache Spark avec PySpark

## MASTER D’EXCELLENCE : ADIA / IL / IISE

### Objectifs


- Comprendre le fonctionnement de **SparkContext**
- Manipuler les **RDD**
- Différencier les **Transformations** et les **Actions**
- Observer le principe de **Lazy Evaluation**
- Réaliser quelques opérations de **traitement de données distribuées**
- Analyser un **dataset réel** avec Spark

---

## Structure du TP

<ul>
  <li>Partie 1 : SparkContext</li>
  <li>Partie 2 : Manipulation des RDD</li>
  <li>Partie 3 : Transformations et Actions</li>
  <li>Partie 4 : Lazy Evaluation</li>
  <li>Partie 5 : WordCount</li>
  <li>Atelier final : Analyse d’un dataset </li>
</ul>

##  Partie 1 : Comprendre SparkContext

### Rappel

Dans Spark, le programme Python que vous exécutez dans le notebook joue le rôle de **driver**.

Le driver communique avec le cluster Spark grâce à **SparkContext**.

Architecture simplifiée :

```text
Notebook (Driver)
        ↓
   SparkContext
        ↓
   Spark Master
        ↓
   Spark Worker(s)
```

### Rôle de SparkContext

SparkContext permet de :

- se connecter au cluster Spark
- créer des RDD
- envoyer les tâches vers les workers
- récupérer les résultats

### Consigne

Exécutez les cellules suivantes et observez :
- la version de Spark
- l'adresse du master
- l'interface Spark UI

In [2]:
# Importe la classe SparkSession, point d'entrée principal pour travailler avec
# les DataFrames, le SQL et les APIs structurées d'Apache Spark.
from pyspark.sql import SparkSession

# Initialise (ou récupère) une session Spark via le pattern "Builder"
spark = (
    SparkSession.builder
    # Définit le nom de l'application (visible dans la Spark Web UI et les logs)
    .appName("TP-Spark-Intro")
    # Indique l'adresse du gestionnaire de cluster (ici un cluster Spark Standalone
    # accessible via le nom d'hôte "spark-master" sur le port 7077, typique d'un réseau Docker)
    .master("spark://spark-master:7077")
    # Crée la SparkSession si aucune n'est active, ou retourne l'instance existante
    .getOrCreate()
)

# Récupère le SparkContext sous-jacent, qui gère la connexion bas niveau au cluster,
# l'allocation des ressources et les opérations de type RDD.
sc = spark.sparkContext

# Affiche la version d'Apache Spark exécutée par la session
print("Version Spark :", sc.version)

# Affiche l'URL du Master effectivement configurée et utilisée
print("Master :", sc.master)

Version Spark : 3.5.1
Master : spark://spark-master:7077


### **Ce qu'il faut retenir sur ces lignes**

* **`SparkSession` vs `SparkContext`** : `SparkSession` est l'API moderne et unifiée (depuis Spark 2.0) pour manipuler des tables et DataFrames ; `spark.sparkContext` (`sc`) reste accessible lorsque tu as besoin d'accéder aux fonctionnalités bas niveau ou aux configurations du contexte Spark.
* **`.master("spark://spark-master:7077")`** : au lieu d'exécuter Spark en local (`local[*]`), ce paramètre connecte ton driver à un **Spark Master** distant (souvent un autre conteneur dans un `docker-compose`), qui répartira ensuite les tâches sur les *Workers*.
* **`.getOrCreate()`** : garantit qu'une seule `SparkSession` est active par processus Python, évitant les conflits de configuration ou de contexte.

### Questions

1. Quelle est la version de Spark utilisée ?
2. Quelle est l’adresse du master ?
3. Ouvrez l’interface Spark UI et vérifiez qu’un worker est bien connecté.

### Interfaces utiles

- **Jupyter / Notebook** : `http://localhost:8888`
- **Spark Master UI** : `http://localhost:8080`
- **Spark Application UI** : `http://localhost:4040`

##  Partie 2 : Manipulation des RDD

### Définition

Un **RDD (Resilient Distributed Dataset)** est une collection de données :

- **distribuée**
- **tolérante aux pannes**
- **traitée en parallèle**

En pratique, un RDD est découpé en **partitions**.

Chaque partition peut être traitée par une **task** Spark.

In [4]:
# Crée une liste Python locale contenant 5 entiers. À ce stade, les données sont
# uniquement en mémoire dans le processus Python du driver.
data = [1, 2, 3, 4, 5]

# Distribue (parallélise) la liste "data" sur le cluster Spark pour créer un RDD
# (Resilient Distributed Dataset) découpé en 4 partitions logiques.
rdd = sc.parallelize(data, 4)

# Action : rapatrie tous les éléments de toutes les partitions vers le driver sous
# forme d'une liste Python (à utiliser avec prudence sur de très gros volumes).
print("Contenu du RDD :", rdd.collect())

# Action : déclenche un job Spark pour compter le nombre total d'éléments
# présents dans l'ensemble des partitions du RDD (ici : 5).
print("Nombre d'éléments :", rdd.count())

# Action : récupère et retourne uniquement le premier élément du RDD (ici : 1),
# sans forcément lire toutes les partitions.
print("Premier élément :", rdd.first())

# Méthode d'inspection : retourne le nombre de partitions du RDD (ici : 4).
print("Nombre de partitions :", rdd.getNumPartitions())

Contenu du RDD : [1, 2, 3, 4, 5]
Nombre d'éléments : 5
Premier élément : 1
Nombre de partitions : 4


### **Points clés à retenir**

* **Évaluation paresseuse (*Lazy Evaluation*) :** `sc.parallelize(...)` prépare la structure distribuée, mais ce sont les **actions** (`collect()`, `count()`, `first()`) qui déclenchent réellement l'exécution des tâches sur les executors.
* **Découpage en 4 partitions (`numSlices=4`) :** les 5 éléments sont répartis sur 4 partitions (par exemple `[1]`, `[2]`, `[3]`, `[4, 5]`). Le nombre de partitions détermine le **niveau de parallélisme potentiel** (au maximum 1 tâche exécutée en parallèle par partition disponible sur les executors).
* **Attention à `.collect()` :** comme il ramène toutes les données dans la mémoire du *driver*, il est parfait pour un petit `TP` ou du debug, mais risqué sur un gros dataset en production (risque d'*OutOfMemoryError* côté driver).

### Exercice 1

Créer un RDD contenant les valeurs suivantes :

```python
[10, 20, 30, 40, 50, 60]
```

Puis :
1. afficher son contenu,
2. afficher son nombre d’éléments,
3. afficher son premier élément.

In [6]:
# Crée un RDD à partir d'une liste de 6 entiers. Comme le nombre de partitions
# n'est pas précisé ici, Spark choisit un nombre de partitions par défaut
# (généralement lié au nombre de cœurs/threads disponibles).
numbers = sc.parallelize([10, 20, 30, 40, 50, 60])

# Action : déclenche un job Spark et rassemble tous les éléments de toutes les
# partitions vers le driver sous forme de liste Python -> [10, 20, 30, 40, 50, 60].
print(numbers.collect())

# Action : déclenche un job Spark pour compter le nombre total d'éléments
# dans le RDD -> 6.
print(numbers.count())

# Action : récupère et affiche uniquement le premier élément du RDD -> 10.
print(numbers.first())

[10, 20, 30, 40, 50, 60]
6
10


### **Ce qu'il faut remarquer ici**

* **Nombre de partitions automatique :** contrairement à `sc.parallelize(data, 4)` où le nombre de partitions était fixé à `4`, omettre le second argument laisse Spark appliquer sa valeur par défaut selon la configuration du contexte/cluster.
* **Trois actions consécutives :** `collect()`, `count()` et `first()` sont toutes des **actions** Spark — chacune lance l'exécution du calcul sur le cluster pour produire un résultat renvoyé au *driver*.

### Atelier 1 : Visualiser les partitions

Créer un RDD avec **4 partitions** et observer comment Spark répartit les données.

In [7]:
# Crée un RDD à partir de la séquence d'entiers de 1 à 10 (range(1, 11)),
# répartie explicitement sur 4 partitions logiques.
rdd_part = sc.parallelize(range(1, 11), 4)

# Méthode d'inspection : retourne le nombre total de partitions du RDD (ici : 4).
print("Nombre de partitions :", rdd_part.getNumPartitions())

# Transformation + Action :
# - .glom() regroupe les éléments de CHAQUE partition dans une liste Python
#   (produisant un RDD de listes, une liste par partition).
# - .collect() déclenche l'exécution et rapatrie ces listes vers le driver.
print("Contenu des partitions :", rdd_part.glom().collect())

Nombre de partitions : 4
Contenu des partitions : [[1, 2], [3, 4, 5], [6, 7], [8, 9, 10]]


### **Comment Spark répartit les 10 éléments sur 4 partitions ?**

Avec 10 éléments (`1` à `10`) et `4` partitions, la répartition suit le découpage interne de `parallelize` (souvent proche de `[[1, 2], [3, 5]` ou `[[1, 2], [3, 4, 5], [6, 7], [8, 9, 10]]` selon la logique d'indices de `sc.parallelize` : en pratique, pour `range(1, 11)` sur 4 slices, les bornes sont `0, 2, 5, 7, 10`, ce qui donne `[[1, 2], [3, 4, 5], [6, 7], [8, 9, 10]]`).

* **Pourquoi `glom()` est très utile en TP :** contrairement à `rdd_part.collect()` qui « aplatit » tout en une seule liste `[1, 2, ..., 10]`, `.glom().collect()` te montre **exactement quel sous-ensemble de données se trouve dans chaque partition**, ce qui aide à comprendre le parallélisme et les opérations par partition (`mapPartitions`).
* **Attention au volume :** comme `collect()`, `.glom().collect()` ramène tout sur le *driver* — à réserver à l'exploration ou à de petits jeux de données.

### Interprétation

Si vous obtenez par exemple :

```python
[[1, 2], [3, 4], [5, 6], [7, 8, 9, 10]]
```

cela signifie :

- partition 0 → `[1, 2]`
- partition 1 → `[3, 4]`
- partition 2 → `[5, 6]`
- partition 3 → `[7, 8, 9, 10]`

Dans Spark, on peut retenir la règle :

> **1 partition = 1 task**

## Partie 3 : Différencier Transformations et Actions

### 1. Transformations

Les transformations créent un **nouveau RDD** sans déclencher immédiatement le calcul.

Exemples :

- `map()`
- `filter()`
- `flatMap()`

### 2. Actions

Les actions déclenchent réellement l'exécution du calcul.

Exemples :

- `collect()`
- `count()`
- `first()`
- `take()`

In [9]:
# Crée un RDD contenant les entiers de 0 à 9 (range(10)), répartis sur le nombre
# de partitions par défaut. Aucun calcul distribué n'est encore exécuté.
rdd = sc.parallelize(range(10))
print("Contenu des partitions :", rdd.glom().collect())

# Transformation (narrow / étroite) : applique la fonction x -> x * 2 à chaque
# élément de chaque partition indépendamment pour produire un nouveau RDD
# contenant [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]. Toujours en lazy evaluation.
rdd2 = rdd.map(lambda x: x * 2)
print("Contenu des partitions :", rdd2.glom().collect())

# Transformation (narrow / étroite) : ne conserve que les éléments strictement
# supérieurs à 10 sur chaque partition indépendamment -> [12, 14, 16, 18].
# Toujours différé jusqu'à la rencontre d'une action.
rdd3 = rdd2.filter(lambda x: x > 10)
print("Contenu des partitions :", rdd3.glom().collect())

# Action : déclenche réellement l'exécution du graphe (DAG) sur le cluster et
# rapatrie le résultat final vers le driver -> [12, 14, 16, 18].
print("Résultat final :", rdd3.collect())

Contenu des partitions : [[0, 1, 2, 3, 4], [5, 6, 7, 8, 9]]
Contenu des partitions : [[0, 2, 4, 6, 8], [10, 12, 14, 16, 18]]
Contenu des partitions : [[], [12, 14, 16, 18]]
Résultat final : [12, 14, 16, 18]


### **Ce qu'il se passe sous le capot (DAG & *Pipelining*)**

* **Transformations étroites (*Narrow Transformations*) :** `map()` et `filter()` n'ont pas besoin de déplacer des données entre partitions (pas de *shuffle*). Chaque partition peut être traitée de manière autonome.
* **Chaînage des opérations (*Pipelining*) :** tant que `.collect()` n'est pas appelé, Spark enregistre simplement les étapes dans son **DAG** (*Directed Acyclic Graph*). Au moment de `.collect()`, Spark peut même fusionner `map` puis `filter` dans une même passe par élément sur chaque partition, sans matérialiser intermédiairement tous les éléments inutiles.
* **Résultat attendu :** les entiers de `0` à `9` multipliés par `2` donnent `0` à `18` ; seuls `12, 14, 16, 18` sont strictement supérieurs à `10`.

### Exercice 2

À partir d’un RDD contenant les nombres de 1 à 15 :

1. multiplier chaque valeur par 3,
2. conserver seulement les valeurs supérieures à 20,
3. afficher le résultat.

In [10]:
# Votre code ici
rdd_ex2 = sc.parallelize(range(1, 16))

result_ex2 = (
    rdd_ex2
    .map(lambda x: x * 3)
    .filter(lambda x: x > 20)
)

print(result_ex2.collect())

[21, 24, 27, 30, 33, 36, 39, 42, 45]


### Atelier 2 : Lire Spark UI

Après exécution d’un calcul, ouvrez :

`http://localhost:4040`

Puis observez :

- **Jobs**
- **Stages**
- **Tasks**
- **Executors**

### À retenir

Organisation interne de Spark :

```text
Job
 └── Stage
      └── Tasks
```

Dans Apache Spark, l'exécution d'un calcul distribué suit une hiérarchie stricte en trois niveaux : **Job → Stage → Task**. Comprendre cette organisation te permet de lire la Spark Web UI, d'anticiper les coûts réseau (*shuffle*) et d'optimiser le parallélisme de tes traitements.

---

### **1. Le Job (déclenché par une Action)**

Un **Job** correspond à l'ensemble des calculs nécessaires pour produire le résultat d'une **action** Spark (`collect()`, `count()`, `first()`, `show()`, `saveAsTextFile()`, etc.).

* **Comment il naît :** tant que tu n'appelles que des transformations (`map()`, `filter()`, `groupByKey()`, `reduceByKey()`, `join()`), Spark construit uniquement un **DAG** (*Directed Acyclic Graph*, le graphe des dépendances entre RDD/DataFrames) en *lazy evaluation*. Dès qu'une action est appelée, le *DAGScheduler* découpe le graphe en un **Job**.
* **Combien y a-t-il de Jobs ?** En règle générale, **1 action = 1 Job** (certaines actions particulières comme `take()` peuvent déclencher plusieurs petits jobs itératifs). Dans ton exemple précédent avec `rdd3.collect()`, un seul Job est créé.

---

### **2. Le Stage (découpé par les *Shuffle Boundaries*)**

Un **Stage** (étape) est un ensemble de transformations étroites (*narrow transformations*) qui peuvent être exécutées **en chaîne (*pipelining*)** sans déplacer de données entre les nœuds du cluster.

* **Transformations étroites (*Narrow*) :** chaque partition parent alimente au maximum une partition enfant (`map`, `filter`, `flatMap`). Elles restent dans le **même Stage**.
* **Transformations larges (*Wide*) et *Shuffle* :** lorsqu'une opération nécessite de regrouper ou répartir des données selon une clé (`groupByKey`, `reduceByKey`, `join`, `repartition`), les partitions d'un stage doivent être redistribuées sur le réseau vers d'autres partitions : c'est le **Shuffle** (avec écriture/lecture de fichiers intermédiaires *shuffle files*).
* **Règle fondamentale :** chaque **Shuffle** crée une frontière de Stage.
* S'il n'y a aucun *shuffle* (comme `sc.parallelize(range(10)).map(...).filter(...).collect()`), le Job ne comporte qu'**un seul Stage**.
* S'il y a un *shuffle* (par exemple un `reduceByKey` suivi d'un `collect`), tu auras **au moins 2 Stages** : un *ShuffleMapStage* (qui prépare et écrit les données *shuffle*) et un *ResultStage* (qui consomme les données et produit le résultat final).



---

### **3. La Task (l'unité élémentaire d'exécution)**

Une **Task** est la plus petite unité de travail envoyée par le *TaskScheduler* à un **Executor** (sur un *Worker*) pour être exécutée sur **une seule partition de données**.

* **Relation Task ↔ Partition :** dans un Stage donné, **1 partition = 1 Task** (en l'absence de *coalesce* ou cas particulier). Si un Stage opère sur un RDD de `4` partitions, ce Stage générera `4` Tasks exécutées en parallèle selon le nombre de *cores* alloués à tes executors.
* **Cycle de vie :** chaque Task exécute le même code (les transformations *pipelinées* du Stage), mais chacun sur un morceau différent des données. Si une Task échoue, Spark peut la rejouer individuellement grâce au lignage (*lineage*) du RDD sans relancer tout le Job.

---

### **Récapitulatif comparatif**

| Niveau | Déclencheur / Limite | Parallélisme / Granularité | Où ça se décide / s'exécute |
| --- | --- | --- | --- |
| **Job** | Appel d'une **Action** (`collect()`, `count()`, etc.) | 1 Job par action principale | Planifié par le **DAGScheduler** sur le *Driver* |
| **Stage** | Frontière de **Shuffle** (*wide transformation*) | Découpé en étapes séquentielles selon les dépendances *shuffle* | Calculé par le **DAGScheduler** sur le *Driver* |
| **Task** | **1 partition** du Stage courant | `N` Tasks en parallèle (`N` = nombre de partitions du Stage) | Envoyée et exécutée sur un *core* d'un **Executor** |

### **Exemple concret pour visualiser**

Imagine ce pipeline :

1. `sc.parallelize(data, 4)` → 4 partitions.
2. `.map(...)` puis `.filter(...)` → transformations étroites (*Stage 1*).
3. `.reduceByKey(...)` → transformation large (*Shuffle* ! Fin du *Stage 1*, début du *Stage 2*).
4. `.collect()` → action qui déclenche le **Job**.

Sous le capot, Spark crée **1 Job**, divisé en **2 Stages**, chacun composé de **4 Tasks** (une par partition), distribuées sur les *cores* disponibles de tes *Executors*.

##  Partie 4 : Observer le principe de Lazy Evaluation

### Définition

Spark n’exécute pas immédiatement les transformations.

Il construit d’abord un **plan d’exécution**.

Le calcul réel commence uniquement lorsqu’une **action** est appelée.

C’est ce qu’on appelle :

> **Lazy Evaluation**

In [11]:
# Crée un RDD contenant les entiers de 0 à 9 découpés en 4 partitions.
# Aucun calcul distribué n'est lancé ici : le plan d'exécution est simplement préparé.
rdd_lazy = sc.parallelize(range(10), 4)

# Transformation (narrow) : ajoute l'étape "multiplier chaque élément par 2" dans le
# graphe de dépendances (lineage / DAG). Toujours aucune exécution sur les executors.
rdd_step1 = rdd_lazy.map(lambda x: x * 2)

# Transformation (narrow) : ajoute l'étape "filtrer les éléments > 10" à la suite
# du DAG. Comme map et filter sont des transformations étroites (sans shuffle),
# elles pourront être regroupées dans un seul Stage lors de l'exécution.
rdd_step2 = rdd_step1.filter(lambda x: x > 10)

# Affiche un message côté driver : à ce stade, le DAG existe en mémoire sur le
# driver, mais aucun Job, Stage ou Task n'a encore été soumis au cluster.
print("Les transformations ont été définies.")

# Confirmation explicite : tant qu'aucune action (collect, count, first, etc.)
# n'est appelée sur rdd_step2, Spark ne calcule rien du tout.
print("Aucune action n'a encore été déclenchée.")

Les transformations ont été définies.
Aucune action n'a encore été déclenchée.


### Observation

À ce stade, ouvrez Spark UI (`http://localhost:4040`) :

- vous ne verrez pas de job significatif lié à un calcul déclenché par `collect()`,
- car aucune action finale n’a encore été demandée.

Maintenant, déclenchez le calcul :

In [12]:
print(rdd_step2.collect())

[12, 14, 16, 18]


### Exercice 3

Créez un RDD contenant les nombres de 1 à 20, puis :

1. ajoutez 5 à chaque valeur,
2. gardez les valeurs paires,
3. n’exécutez pas tout de suite d’action,
4. observez Spark UI,
5. enfin, déclenchez l’exécution avec `collect()`.

In [13]:
# Votre code ici
rdd_ex3 = sc.parallelize(range(1, 21), 4)
rdd_ex3_t = rdd_ex3.map(lambda x: x + 5).filter(lambda x: x % 2 == 0)

# Décommentez la ligne suivante pour déclencher l'action :
print(rdd_ex3_t.collect())

[6, 8, 10, 12, 14, 16, 18, 20, 22, 24]


### Atelier 3 : Visualiser le parallélisme

Le code suivant ralentit volontairement le traitement pour vous laisser le temps d’observer les **tasks** dans Spark UI.

In [14]:
import time

def slow(x):
    time.sleep(2)
    return x * 2

rdd_slow = sc.parallelize(range(20), 4)
result_slow = rdd_slow.map(slow).collect()

print(result_slow)

[Stage 21:=============================>                            (2 + 2) / 4]

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38]


### Questions

1. Combien de tasks voyez-vous dans Spark UI ?
2. Ce nombre correspond-il au nombre de partitions ?
3. Pourquoi Spark ne lance-t-il le calcul qu’au moment de `collect()` ?

## Partie 5 : Réaliser un WordCount avec les RDD

Le problème WordCount est un grand classique en Big Data.

Il consiste à compter le nombre d’occurrences de chaque mot dans un ensemble de phrases.

In [18]:
# Crée un RDD contenant 3 chaînes de caractères (réparties sur le nombre de
# partitions par défaut de Spark). Toujours en évaluation paresseuse (lazy).
lines = sc.parallelize([
    "big data is powerful",
    "spark is fast",
    "big data needs spark"
])
print('Partition lines => ', lines.glom().collect())

# Transformation étroite (narrow) : découpe chaque ligne en mots (line.split())
# puis "aplatit" le résultat pour obtenir un seul RDD de mots (1 élément = 1 mot),
# au lieu d'une liste de listes.
words = lines.flatMap(lambda line: line.split())
print('Partition words=> ', words.glom().collect())

# Transformation étroite (narrow) : transforme chaque mot en paire clé-valeur
# (mot, 1), format requis pour les opérations par clé comme reduceByKey.
pairs = words.map(lambda word: (word, 1))
print('Partition pairs=> ', pairs.glom().collect())

# Transformation large (wide) : regroupe les valeurs associées à une même clé (word)
# et les additionne (a + b) localement puis globalement. Cela déclenche un SHUFFLE
# entre partitions pour rassembler toutes les occurrences d'un même mot.
counts = pairs.reduceByKey(lambda a, b: a + b)
print('Partition counts=> ', counts.glom().collect())

# Action : déclenche le Job complet et rapatrie le résultat final vers le driver
# (par exemple [('big', 2), ('data', 2), ('is', 2), ('powerful', 1), ('spark', 2), ('fast', 1), ('needs', 1)]).
print(counts.collect())

Partition lines =>  [['big data is powerful'], ['spark is fast', 'big data needs spark']]
Partition words=>  [['big', 'data', 'is', 'powerful'], ['spark', 'is', 'fast', 'big', 'data', 'needs', 'spark']]
Partition pairs=>  [[('big', 1), ('data', 1), ('is', 1), ('powerful', 1)], [('spark', 1), ('is', 1), ('fast', 1), ('big', 1), ('data', 1), ('needs', 1), ('spark', 1)]]
Partition counts=>  [[('big', 2), ('powerful', 1), ('fast', 1), ('needs', 1)], [('data', 2), ('is', 2), ('spark', 2)]]
[('big', 2), ('powerful', 1), ('fast', 1), ('needs', 1), ('data', 2), ('is', 2), ('spark', 2)]


### Explication des étapes

1. `flatMap()` : découpe chaque ligne en mots
2. `map()` : transforme chaque mot en paire `(mot, 1)`
3. `reduceByKey()` : additionne les occurrences
4. `collect()` : récupère le résultat

### Exercice 4

À partir des phrases suivantes :

```python
[
    "python is simple",
    "spark is powerful",
    "python and spark"
]
```

Calculer le nombre d’occurrences de chaque mot.

In [19]:
# Votre code ici
lines_ex4 = sc.parallelize([
    "python is simple",
    "spark is powerful",
    "python and spark"
])

wordcount_ex4 = (
    lines_ex4
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)

print(wordcount_ex4.collect())

[('python', 2), ('powerful', 1), ('and', 1), ('is', 2), ('simple', 1), ('spark', 2)]


## Atelier final : Analyse d’un dataset

### Contexte

Dans cette partie, on traite un fichier CSV contenant des ventes.

Exemple de structure :

```csv
product,category,price,quantity
PC,IT,8000,2
Mouse,IT,150,10
Chair,Furniture,1200,3
Desk,Furniture,2500,2
```

### Objectif

Utiliser Spark pour effectuer quelques analyses simples sur un dataset réel.

### Étape 1 : Charger le fichier

Placez votre fichier CSV dans le dossier partagé du conteneur, puis chargez-le avec `textFile()`.

> Adaptez le chemin selon votre environnement.

In [22]:
# Définit le chemin vers le fichier CSV (ici dans /opt/spark-data/sales.csv,
# chemin qui doit être accessible depuis le driver et les workers, par exemple
# via un volume Docker monté dans les conteneurs Spark).
file_path = "/opt/spark-data/sales.csv"

# Transformation (lazy) : lit le fichier ligne par ligne sous forme d'un RDD de
# chaînes de caractères (chaque élément du RDD = une ligne du fichier CSV,
# y compris l'en-tête s'il y en a un). Spark découpe la lecture en partitions
# (selon la taille des blocs/splits et le parallélisme par défaut).
data_csv = sc.textFile(file_path)

# Action : déclenche un Job pour lire et afficher la toute première ligne du
# fichier (souvent l'en-tête des colonnes, ex. "order_id,product,amount,...").
print("Première ligne :", data_csv.first())

# Action : déclenche un second Job pour parcourir toutes les partitions et
# compter le nombre total de lignes (en-tête incluse si présente).
print("Nombre total de lignes :", data_csv.count())

Première ligne : product,category,price,quantity
Nombre total de lignes : 5


### Étape 2 : Supprimer l’en-tête et parser les lignes

In [24]:
# Action : déclenche un Job pour récupérer la première ligne du RDD data_csv
# (par exemple "order_id,product,quantity,price,date") afin de l'identifier comme en-tête.
header = data_csv.first()

# Transformation étroite (lazy) : conserve uniquement les lignes différentes de
# l'en-tête. Cela exclut la ligne de noms de colonnes avant les traitements suivants.
rows = data_csv.filter(lambda line: line != header)

# Transformation étroite (lazy) : convertit chaque ligne texte (str) en une liste
# Python de chaînes de caractères en découpant sur la virgule (",").
parsed = rows.map(lambda line: line.split(","))

# Action : déclenche un Job pour récupérer et renvoyer au driver les 5 premiers
# éléments (5 premières lignes parsées sous forme de listes), sans ramener tout
# le fichier.
print(parsed.take(5))

[['PC', 'IT', '8000', '2'], ['Mouse', 'IT', '150', '10'], ['Chair', 'Furniture', '1200', '3'], ['Desk', 'Furniture', '2500', '2']]


### Étape 3 : Calculer le chiffre d’affaires par produit

In [25]:
# Transformation étroite (lazy) : pour chaque ligne "x" (une liste de chaînes issue
# de line.split(",")), construit une paire clé-valeur :
# - clé   : x[0] (par ex. le nom ou l'identifiant du produit)
# - valeur: produit numérique de x[2] (converti en float) et x[3] (converti en int),
#           par exemple un prix unitaire * une quantité.
# Toujours aucune exécution tant qu'une action n'est pas appelée.
sales_by_product = parsed.map(lambda x: (x[0], float(x[2]) * int(x[3])))

# Action : déclenche le Job Spark (enchaînant les étapes précédentes sur les
# partitions) et rapatrie toutes les paires (clé, montant) vers le driver.
print(sales_by_product.collect())

[('PC', 16000.0), ('Mouse', 1500.0), ('Chair', 3600.0), ('Desk', 5000.0)]


### Étape 4 : Calculer le chiffre d’affaires par catégorie

In [27]:
# Chaîne deux transformations sur le RDD "parsed" (toujours en lazy evaluation) :
sales_by_category = (
    # 1. Transformation étroite (narrow) : crée une paire clé-valeur où :
    #    - la clé est x[1] (ex. la catégorie du produit)
    #    - la valeur est le montant de la ligne (float(x[2]) * int(x[3]))
    parsed
    .map(lambda x: (x[1], float(x[2]) * int(x[3])))
    # 2. Transformation large (wide) : regroupe toutes les valeurs associées à une
    #    même catégorie (x[1]) et les additionne (a + b). Cela déclenche un SHUFFLE
    #    entre partitions et crée une frontière de Stage.
    .reduceByKey(lambda a, b: a + b)
)

# Action : déclenche le Job Spark sur le cluster et rapatrie vers le driver
# la liste finale des paires (categorie, chiffre_affaires_total).
print(sales_by_category.collect())

[('IT', 17500.0), ('Furniture', 8600.0)]


### Étape 5 : Calculer le chiffre d’affaires total

In [28]:
# Enchaîne une transformation étroite et une action d'agrégation globale :
total_revenue = (
    # 1. Transformation étroite (narrow / lazy) : calcule le montant de chaque
    #    ligne sous forme d'un float (ex. prix * quantité), sans clé associée.
    parsed
    .map(lambda x: float(x[2]) * int(x[3]))
    # 2. Action : déclenche le Job Spark, calcule la somme locale dans chaque
    #    partition (une Task par partition), puis additionne les sommes partielles
    #    sur le driver pour retourner un unique nombre (scalaire).
    .sum()
)

# Affiche le chiffre d'affaires total directement sur le driver
print("Chiffre d'affaires total :", total_revenue)

Chiffre d'affaires total : 26100.0


##  Exercice final guidé

À partir du dataset réel, répondre aux questions suivantes :

1. Combien de lignes contient le dataset ?
2. Quel est le chiffre d’affaires total ?
3. Quelle catégorie génère le plus de revenus ?
4. Quel produit a généré le chiffre d’affaires le plus élevé ?
5. Combien de catégories différentes existe-t-il dans le dataset ?

### Indication

Utilisez :

- `count()`
- `map()`
- `reduceByKey()`
- `collect()`
- `distinct()`

In [29]:
# 1. Combien de lignes contient le dataset ?
print(f"Number of lines => {rows.count()}")

Number of lines => 4


In [34]:
# 2. Quel est le chiffre d’affaires total ?
total_amount = (
    rows
    .map(lambda x: x.split(','))
    .map(lambda x: float(x[2]) * int(x[3]))
    .sum()
)

print(f"Total CA {total_amount}")

Total CA 26100.0


In [55]:
# 3. Quelle catégorie génère le plus de revenus ?
high_category = (
    rows
    .map(lambda x: x.split(','))
    .map(lambda x: (x[1], float(x[2]) * int(x[3])))
    .reduceByKey(lambda a, b: a + b)
    .max(key=lambda x: x[1])
    
)

print(f'resulta => {high_category}')
print(high_category[0])

resulta => ('IT', 17500.0)
IT


In [54]:
# 3. Quelle catégorie génère le plus de revenus ?
high_category = (
    rows
    .map(lambda x: x.split(','))
    .map(lambda x: (x[1], float(x[2]) * int(x[3])))
    .reduceByKey(lambda a, b: a + b)
    .top(1, key=lambda x: x[1])
    
)

print(f'resulta => {high_category}')
high_category[0][0]

resulta => [('IT', 17500.0)]


'IT'

In [56]:
# 4. Quel produit a généré le chiffre d’affaires le plus élevé ?

product = (
    rows
    .map(lambda x: x.split(','))
    .map(lambda x: (x[0], (float(x[2]) * int(x[3]))))
    .reduceByKey(lambda a, b: a + b)
    .top(1, key=lambda x: x[1])
    
)

product[0][0]

'PC'

In [60]:
# 5. Combien de catégories différentes existe-t-il dans le dataset ?

category = (
    rows
    .map(lambda x: x.split(','))
    .map(lambda x: x[0])
    .distinct()
)

category.collect()

['PC', 'Mouse', 'Chair', 'Desk']